# Clustering Algorithms

Unsupervised learning methods for discovering natural groupings in data. Covers K-Means, Hierarchical Clustering, DBSCAN, and evaluation metrics.

## What is Clustering?

Clustering is the task of dividing data points into groups (clusters) such that:
- Points **within** a cluster are similar to each other
- Points **across** different clusters are dissimilar

### Common Applications
- Customer segmentation (marketing)
- Anomaly detection
- Image compression
- Document organization
- Bioinformatics (gene sequence analysis)

## K-Means Clustering

**Algorithm**:
1. Choose K initial centroids (randomly or via K-Means++)
2. Assign each point to the nearest centroid
3. Recompute centroids as the mean of assigned points
4. Repeat steps 2-3 until convergence

**Inertia**: Sum of squared distances of samples to their closest cluster center (minimized by K-Means).

**Elbow Method**: Plot inertia vs K — choose K at the "elbow" where improvement slows.

**Silhouette Score**: Measures how similar a point is to its own cluster vs others (range: -1 to 1).

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style('whitegrid')
print("Libraries imported.")

In [ ]:
# Generate synthetic dataset for clustering
X_blobs, y_blobs = make_blobs(
    n_samples=500, n_features=2, centers=4, cluster_std=1.5, random_state=42
)

plt.figure(figsize=(8, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.7)
plt.title('Synthetic Blobs — True Clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(label='True cluster')
plt.show()

In [ ]:
# K-Means clustering
kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X_blobs)

print("K-Means Results:")
print(f"  Inertia: {kmeans.inertia_:.2f}")
print(f"  Silhouette score: {silhouette_score(X_blobs, kmeans_labels):.4f}")
print(f"  Davies-Bouldin index: {davies_bouldin_score(X_blobs, kmeans_labels):.4f}")
print(f"  Adjusted Rand Index (vs true): {adjusted_rand_score(y_blobs, kmeans_labels):.4f}")

In [ ]:
# Visualize K-Means clustering
plt.figure(figsize=(8, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            s=200, marker='X', c='red', edgecolors='black', label='Centroids')
plt.title('K-Means Clustering (K=4)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.colorbar(label='Cluster')
plt.show()

In [ ]:
# Elbow Method — finding optimal K
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(alpha=0.3)
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 (true)')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'rs-', linewidth=2)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher is better)')
axes[1].grid(alpha=0.3)
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 (true)')
axes[1].legend()

plt.tight_layout()
plt.show()

## Hierarchical Clustering (Agglomerative)

Builds a hierarchy of clusters using a bottom-up approach:
1. Start with each point as its own cluster
2. Repeatedly merge the two closest clusters
3. Stop when only one cluster remains (or K is reached)

**Linkage criteria**:
- **Single**: minimum distance between clusters (can create chains)
- **Complete**: maximum distance between clusters (compact, balanced)
- **Average**: average distance between clusters
- **Ward**: minimizes variance within merged clusters (most popular)

In [ ]:
# Hierarchical Clustering Dendrogram
plt.figure(figsize=(12, 5))
Z = linkage(X_blobs[:100], method='ward')
dendrogram(Z, truncate_mode='level', p=5)
plt.title('Hierarchical Clustering Dendrogram (Ward linkage)')
plt.xlabel('Sample index')
plt.ylabel('Distance')
plt.axhline(y=30, color='red', linestyle='--', label='Cut at distance=30 -> 4 clusters')
plt.legend()
plt.show()

In [ ]:
# AgglomerativeClustering
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_labels = agg.fit_predict(X_blobs)

print("Agglomerative Clustering Results:")
print(f"  Silhouette score: {silhouette_score(X_blobs, agg_labels):.4f}")
print(f"  Adjusted Rand Index: {adjusted_rand_score(y_blobs, agg_labels):.4f}")

In [ ]:
# Compare linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
for method in linkage_methods:
    agg_temp = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels_temp = agg_temp.fit_predict(X_blobs)
    sil = silhouette_score(X_blobs, labels_temp)
    ari = adjusted_rand_score(y_blobs, labels_temp)
    print(f"  {method:10s}  Silhouette={sil:.4f}  ARI={ari:.4f}")

## DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

Groups points that are closely packed together, marking points in low-density regions as outliers.

**Key Parameters**:
- **eps**: maximum distance between two points to be considered neighbors
- **min_samples**: minimum number of points to form a dense region

**Advantages**:
- Does NOT require specifying number of clusters
- Can find arbitrarily shaped clusters
- Robust to outliers (identifies noise points)

**Disadvantages**:
- Sensitive to eps and min_samples
- Struggles with varying density clusters

In [ ]:
# DBSCAN on moons dataset (non-spherical clusters)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)

n_clusters = len(set(dbscan_labels) - {-1})
n_noise = np.sum(dbscan_labels == -1)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_moons[:, 0], X_moons[:, 1], c=dbscan_labels, cmap='viridis',
                       alpha=0.7, edgecolors='k')
plt.title(f'DBSCAN Clustering ({n_clusters} clusters, {n_noise} noise points)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(scatter, label='Cluster label (-1 = noise)')
plt.show()

In [ ]:
# Compare clustering methods on Iris
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Scale
scaler = StandardScaler()
X_iris_s = scaler.fit_transform(X_iris)

# K-Means
km_iris = KMeans(n_clusters=3, n_init=10, random_state=42)
km_labels = km_iris.fit_predict(X_iris_s)

# Agglomerative
agg_iris = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels_iris = agg_iris.fit_predict(X_iris_s)

# DBSCAN
db_iris = DBSCAN(eps=0.5, min_samples=5)
db_labels_iris = db_iris.fit_predict(X_iris_s)

print("Clustering Methods on Iris:")
print(f"  {'Method':15s}  {'Silhouette':12s}  {'ARI vs True':12s}")
print(f"  {'-'*40}")
print(f"  {'K-Means':15s}  {silhouette_score(X_iris_s, km_labels):.4f}         "
      f"{adjusted_rand_score(y_iris, km_labels):.4f}")
print(f"  {'Agglomerative':15s}  {silhouette_score(X_iris_s, agg_labels_iris):.4f}         "
      f"{adjusted_rand_score(y_iris, agg_labels_iris):.4f}")
print(f"  {'DBSCAN':15s}  {silhouette_score(X_iris_s, db_labels_iris):.4f}         "
      f"{adjusted_rand_score(y_iris, db_labels_iris):.4f}")

In [ ]:
# Problem: Customer Segmentation with K-Means
# Generate synthetic customer data
np.random.seed(42)
n_customers = 500

customer_data = np.vstack([
    # Segment 1: Young, low income, high online
    np.random.multivariate_normal([25, 30000, 40], [[16, 0, 0], [0, 25e6, 0], [0, 0, 100]], 150),
    # Segment 2: Middle-aged, mid income, moderate online
    np.random.multivariate_normal([45, 65000, 20], [[25, 0, 0], [0, 50e6, 0], [0, 0, 80]], 200),
    # Segment 3: Older, high income, low online
    np.random.multivariate_normal([60, 100000, 8], [[20, 0, 0], [0, 80e6, 0], [0, 0, 30]], 150),
])

customer_df = pd.DataFrame(customer_data, columns=['Age', 'Income', 'OnlineHours'])
print("Customer data (first 5 rows):")
print(customer_df.head())
print(f"\nShape: {customer_df.shape}")

In [ ]:
# Scale and cluster
scaler_cust = StandardScaler()
X_cust = scaler_cust.fit_transform(customer_df)

# Find optimal K
sil_scores = []
for k in range(2, 8):
    km_temp = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_temp = km_temp.fit_predict(X_cust)
    sil_scores.append(silhouette_score(X_cust, labels_temp))

best_k = range(2, 8)[np.argmax(sil_scores)]
print(f"Optimal K from silhouette: {best_k} (silhouette = {max(sil_scores):.4f})")

# Final model
km_cust = KMeans(n_clusters=best_k, n_init=10, random_state=42)
customer_df['Segment'] = km_cust.fit_predict(X_cust)

print("\nSegment profiles:")
print(customer_df.groupby('Segment').mean().round(1))

In [ ]:
# Visualize customer segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(customer_df['Age'], customer_df['Income'],
                c=customer_df['Segment'], cmap='viridis', alpha=0.6)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Income')
axes[0].set_title('Customer Segments: Age vs Income')

axes[1].scatter(customer_df['Age'], customer_df['OnlineHours'],
                c=customer_df['Segment'], cmap='viridis', alpha=0.6)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Online Hours/Week')
axes[1].set_title('Customer Segments: Age vs Online Hours')

plt.tight_layout()
plt.show()

In [ ]:
# Problem: Hierarchical clustering on Iris with dendrogram
Z_iris = linkage(X_iris_s, method='ward')

plt.figure(figsize=(12, 5))
plt.title('Hierarchical Clustering Dendrogram — Iris Dataset')
dendrogram(Z_iris, labels=y_iris, leaf_font_size=8)
plt.axhline(y=5, color='red', linestyle='--', label='Cut → 3 clusters')
plt.ylabel('Distance')
plt.legend()
plt.show()

In [ ]:
# DBSCAN parameter sensitivity
eps_values = [0.1, 0.2, 0.3, 0.5]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, eps in zip(axes, eps_values):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clusters = len(set(labels) - {-1})
    n_noise = np.sum(labels == -1)
    sc = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', alpha=0.7)
    ax.set_title(f'eps={eps}, clusters={n_clusters}, noise={n_noise}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()
print("Small eps: too many clusters/noise. Large eps: merges clusters.")

In [ ]:
# Interview Problem: Compare K-Means, Hierarchical, and DBSCAN
print("""
Interview Question: Compare K-Means, Hierarchical Clustering, and DBSCAN.

K-Means:
- Pros: Fast, scalable, simple, guaranteed convergence
- Cons: Need K, spherical clusters only, sensitive to outliers
- Best for: Large datasets, spherical clusters, known number of groups

Hierarchical (Agglomerative):
- Pros: No K needed, dendrogram provides rich info, works with non-spherical shapes
- Cons: O(n³) complexity, sensitive to linkage choice
- Best for: Small datasets, when hierarchy is meaningful, exploratory analysis

DBSCAN:
- Pros: No K needed, arbitrary shapes, robust to outliers, identifies noise
- Cons: Sensitive to eps/min_samples, struggles with varying density
- Best for: Non-spherical clusters, noisy data, outlier detection
""")

In [ ]:
# Summary
print("=" * 60)
print("CLUSTERING — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. K-Means: fast, spherical, needs K — use elbow + silhouette
2. Hierarchical: dendrogram visualization, no K needed
3. DBSCAN: density-based, finds arbitrary shapes, identifies noise
4. Always scale features before clustering (distance-based!)
5. Evaluation: silhouette (internal), adjusted Rand index (external)
6. No single best method — depends on data shape and requirements
7. Clustering is UNSUPERVISED — results need human interpretation
""")